In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from IPython.display import display
import warnings

import ast

warnings.filterwarnings('ignore')

# 한글 폰트 설정
plt.rcParams['font.family'] = 'Malgun Gothic'

# 마이너스 기호 깨짐 방지
plt.rcParams['axes.unicode_minus'] = False

# 전역 시드 설정 (재현성을 위해)
np.random.seed(42)

print("="*60)
print("라이브러리 로드 완료!")
print("한글 폰트 설정 완료!")
print("="*60)

라이브러리 로드 완료!
한글 폰트 설정 완료!


## 데이터 로드

In [3]:
pd_df = pd.read_csv('data/products_003.csv')
rv_df = pd.read_csv('data/reviews_003.csv')
print(f'products: {pd_df.shape[0]:,}행 × {pd_df.shape[1]}열')
print(f'reviews: {rv_df.shape[0]:,}행 × {rv_df.shape[1]}열')

products: 73행 × 12열
reviews: 30,767행 × 14열


---

# products

## 1. 기본 구조 확인

In [4]:
print('-- head(5) --')
pd_df.head(5)

-- head(5) --


,플랫폼,카테고리,브랜드,goodsNo,상품명,정가,판매가,할인율(%),조회수,누적판매수,리뷰수,리뷰점수
0,무신사,바지,필루미네이트,3791988,[해피가이정호 pick] 데미지 워시드 데님 팬츠-미디엄 블루,53000,32900,38,73852,149483,16190,96
1,무신사,바지,필루미네이트,3792018,나일론 투 턱 테이핑 팬츠-블랙,58000,39000,33,4078,33588,4448,96
2,무신사,바지,필루미네이트,3791990,[해피가이정호 pick] 데미지 워시드 데님 팬츠-블랙,53000,32900,38,10480,25096,2774,96
3,무신사,바지,필루미네이트,4095000,데미지 워시드 데님 팬츠-스카이블루,53000,34900,34,7499,13091,1642,96
4,무신사,바지,필루미네이트,4969807,[해피가이정호 pick] 올 데이 린넨 라이크 밴딩 팬츠-5Color,49000,32900,33,6510,15076,1393,94


In [5]:
print('-- columns & dtypes --')
pd_df.dtypes

-- columns & dtypes --


플랫폼          str
카테고리         str
브랜드          str
goodsNo    int64
상품명          str
정가         int64
판매가        int64
할인율(%)     int64
조회수        int64
누적판매수      int64
리뷰수        int64
리뷰점수       int64
dtype: object

In [6]:
print('-- info() --')
pd_df.info()

-- info() --
<class 'pandas.DataFrame'>
RangeIndex: 73 entries, 0 to 72
Data columns (total 12 columns):
 #   Column   Non-Null Count  Dtype
---  ------   --------------  -----
 0   플랫폼      73 non-null     str  
 1   카테고리     73 non-null     str  
 2   브랜드      73 non-null     str  
 3   goodsNo  73 non-null     int64
 4   상품명      73 non-null     str  
 5   정가       73 non-null     int64
 6   판매가      73 non-null     int64
 7   할인율(%)   73 non-null     int64
 8   조회수      73 non-null     int64
 9   누적판매수    73 non-null     int64
 10  리뷰수      73 non-null     int64
 11  리뷰점수     73 non-null     int64
dtypes: int64(8), str(4)
memory usage: 7.0 KB


## 2. 기술통계

In [7]:
# 수치형 컬럼 요약 통계
print('-- describe (수치형) --')
pd_df.describe()

-- describe (수치형) --


,goodsNo,정가,판매가,할인율(%),조회수,누적판매수,리뷰수,리뷰점수
count,7.300000e+01,73.000000,73.000000,73.000000,73.000000,73.000000,73.000000,73.000000
mean,5.467390e+06,59726.027397,41145.205479,30.671233,3417.054795,3831.767123,421.465753,72.520548
std,5.952517e+05,8718.212537,5926.752931,6.614493,9347.331347,18132.321791,1986.531875,41.904690
min,3.791988e+06,49000.000000,32900.000000,19.000000,0.000000,0.000000,0.000000,0.000000
25%,5.333965e+06,53000.000000,37900.000000,27.000000,353.000000,0.000000,1.000000,84.000000
50%,5.352337e+06,59000.000000,39900.000000,32.000000,533.000000,57.000000,5.000000,96.000000
75%,5.911825e+06,69000.000000,41900.000000,36.000000,2927.000000,237.000000,30.000000,98.000000
max,6.140880e+06,98000.000000,59900.000000,42.000000,73852.000000,149483.000000,16190.000000,100.000000


In [8]:
# 문자형 컬럼 요약 통계
print('-- describe (object형) --')
pd_df.describe(include='object')

-- describe (object형) --


,플랫폼,카테고리,브랜드,상품명
count,73,73,73,73
unique,1,1,1,73
top,무신사,바지,필루미네이트,[해피가이정호 pick] 데미지 워시드 데님 팬츠-미디엄 블루
freq,73,73,73,1


## 3. 결측치 및 중복 행 확인 

In [9]:
null_df = pd.DataFrame({
    '결측 수': pd_df.isnull().sum(),
    '결측 비율(%)': (pd_df.isnull().sum() / len(pd_df) * 100).round(2)
}).sort_values('결측 비율(%)', ascending=False)

null_df[null_df['결측 수'] > 0]

,결측 수,결측 비율(%)


In [10]:
dup_count = pd_df.duplicated().sum()
print(f'중복 행 수: {dup_count:,}개 ({dup_count / len(pd_df) * 100:.2f}%)')

if dup_count > 0:
    print('\n── 중복 행 샘플 ──')
    display(pd_df[pd_df.duplicated(keep=False)].sort_values(pd_df.columns.tolist()).head(10))

중복 행 수: 0개 (0.00%)


In [11]:
cat_cols = pd_df.select_dtypes(include=['object', 'category']).columns.tolist()
print(f'카테고리 컬럼: {cat_cols}\n')

for col in cat_cols:
    nunique = pd_df[col].nunique(dropna=False)
    print(f'[{col}]  고유값 수: {nunique}')

카테고리 컬럼: ['플랫폼', '카테고리', '브랜드', '상품명']

[플랫폼]  고유값 수: 1
[카테고리]  고유값 수: 1
[브랜드]  고유값 수: 1
[상품명]  고유값 수: 73


## 4. 요약

In [12]:
num_cols = pd_df.select_dtypes(include='number').columns.tolist()
print('=' * 40)
print('  products_003.csv  기초 분석 요약')
print('=' * 40)
print(f'  행(row)          : {pd_df.shape[0]:,}')
print(f'  열(column)       : {pd_df.shape[1]}')
print(f'  수치형 컬럼      : {len(num_cols)}개')
print(f'  카테고리 컬럼    : {len(cat_cols)}개')
print(f'  결측값 있는 컬럼 : {null_df[null_df["결측 수"] > 0].shape[0]}개')
print(f'  전체 결측률      : {pd_df.isnull().mean().mean() * 100:.2f}%')
print(f'  중복 행          : {dup_count:,}개')
print('=' * 40)

  products_003.csv  기초 분석 요약
  행(row)          : 73
  열(column)       : 12
  수치형 컬럼      : 8개
  카테고리 컬럼    : 4개
  결측값 있는 컬럼 : 0개
  전체 결측률      : 0.00%
  중복 행          : 0개


---

# reviews

In [13]:
rv_df2 = rv_df.drop(columns=['리뷰내용'])

## 1. 기본 구조 확인

In [14]:
print('-- head(5) --')
rv_df2.head(5)

-- head(5) --


,goodsNo,리뷰번호,작성자,평점,체험단,구매옵션,키,몸무게,성별,작성일,만족도,사진유무,도움돼요
0,3791988,83692475,사막바다,4,False,S,169,59,여성,2026-04-15T08:56:23.000+09:00,사이즈: 정사이즈 / 화면 대비 색감: 화면과 비슷 / 퀄리티: 보통 / 신축성: 적당함,False,0
1,3791988,83690132,dngur04,5,False,L,183,85,남성,2026-04-15T03:14:08.000+09:00,사이즈: 정사이즈 / 화면 대비 색감: 화면과 비슷 / 퀄리티: 좋음 / 신축성: ...,False,0
2,3791988,83682719,갖고싶은LA집업,5,False,S,178,61,남성,2026-04-14T21:43:19.000+09:00,사이즈: 정사이즈 / 화면 대비 색감: 화면과 비슷 / 퀄리티: 매우 좋음 / 신축...,True,0
3,3791988,83676851,흐르는남색랭킹룩,5,False,S,173,72,남성,2026-04-14T18:54:39.000+09:00,사이즈: 정사이즈 / 화면 대비 색감: 화면과 비슷 / 퀄리티: 보통 / 신축성: ...,True,0
4,3791988,83676828,흐르는남색랭킹룩,5,False,XL,173,72,남성,2026-04-14T18:53:49.000+09:00,사이즈: 정사이즈 / 화면 대비 색감: 화면과 비슷 / 퀄리티: 좋음 / 신축성: ...,True,0


In [15]:
print('-- columns & dtypes --')
rv_df.dtypes

-- columns & dtypes --


goodsNo    int64
리뷰번호       int64
작성자          str
리뷰내용         str
평점         int64
체험단         bool
구매옵션         str
키          int64
몸무게        int64
성별           str
작성일          str
만족도          str
사진유무        bool
도움돼요       int64
dtype: object

In [16]:
print('-- info() --')
rv_df.info()

-- info() --
<class 'pandas.DataFrame'>
RangeIndex: 30767 entries, 0 to 30766
Data columns (total 14 columns):
 #   Column   Non-Null Count  Dtype
---  ------   --------------  -----
 0   goodsNo  30767 non-null  int64
 1   리뷰번호     30767 non-null  int64
 2   작성자      30767 non-null  str  
 3   리뷰내용     30717 non-null  str  
 4   평점       30767 non-null  int64
 5   체험단      30767 non-null  bool 
 6   구매옵션     30637 non-null  str  
 7   키        30767 non-null  int64
 8   몸무게      30767 non-null  int64
 9   성별       30172 non-null  str  
 10  작성일      30767 non-null  str  
 11  만족도      3456 non-null   str  
 12  사진유무     30767 non-null  bool 
 13  도움돼요     30767 non-null  int64
dtypes: bool(2), int64(6), str(6)
memory usage: 2.9 MB


## 2. 기술통계

In [17]:
# 수치형 컬럼 요약 통계
print('-- describe (수치형) --')
rv_df.describe()

-- describe (수치형) --


,goodsNo,리뷰번호,평점,키,몸무게,도움돼요
count,3.076700e+04,3.076700e+04,30767.000000,30767.000000,30767.000000,30767.000000
mean,3.992192e+06,6.956958e+07,4.724185,166.148763,64.995027,0.285598
std,4.382788e+05,7.447576e+06,0.746559,31.103301,33.280341,6.551715
min,3.791988e+06,5.532523e+07,0.000000,0.000000,0.000000,0.000000
25%,3.791988e+06,6.283976e+07,5.000000,165.000000,57.000000,0.000000
50%,3.791988e+06,6.910795e+07,5.000000,172.000000,65.000000,0.000000
75%,3.792018e+06,7.620722e+07,5.000000,177.000000,75.000000,0.000000
max,6.140877e+06,8.371255e+07,5.000000,1172.000000,4850.000000,749.000000


In [18]:
print('-- describe (object형) --')
rv_df.describe(include='object')

-- describe (object형) --


,작성자,리뷰내용,구매옵션,성별,작성일,만족도
count,30767,30717,30637,30172,30767,3456
unique,21785,28696,53,2,30753,223
top,-,재질이 얇아서 여름에도 입기 좋을것같고 색감이 밝아서 시원해보입니다,M,남성,2026-01-04T16:11:24.000+09:00,사이즈: 정사이즈 / 화면 대비 색감: 화면과 비슷 / 퀄리티: 보통 / 신축성: 적당함
freq,89,6,7220,20254,2,1264


## 3. 결측치 및 중복 행 확인 

In [19]:
rv_null_df = pd.DataFrame({
    '결측 수': rv_df.isnull().sum(),
    '결측 비율(%)': (rv_df.isnull().sum() / len(rv_df) * 100).round(2)
}).sort_values('결측 비율(%)', ascending=False)

rv_null_df[rv_null_df['결측 수'] > 0]

,결측 수,결측 비율(%)
만족도,27311,88.77
성별,595,1.93
구매옵션,130,0.42
리뷰내용,50,0.16


In [20]:
rv_dup_count = rv_df.duplicated().sum()
print(f'중복 행 수: {rv_dup_count:,}개 ({rv_dup_count / len(rv_df) * 100:.2f}%)')

if rv_dup_count > 0:
    print('\n── 중복 행 샘플 ──')
    display(rv_df[rv_df.duplicated(keep=False)].sort_values(rv_df.columns.tolist()).head(10))

중복 행 수: 0개 (0.00%)


In [21]:
# 결측 포함 고유값
rv_cat_cols = rv_df.select_dtypes(include=['object', 'category']).columns.tolist()
print(f'카테고리 컬럼: {rv_cat_cols}\n')

for col in rv_cat_cols:
    nunique = rv_df[col].nunique(dropna=False)
    print(f'[{col}]  고유값 수: {nunique}')

카테고리 컬럼: ['작성자', '리뷰내용', '구매옵션', '성별', '작성일', '만족도']

[작성자]  고유값 수: 21785
[리뷰내용]  고유값 수: 28697
[구매옵션]  고유값 수: 54
[성별]  고유값 수: 3
[작성일]  고유값 수: 30753
[만족도]  고유값 수: 224


In [22]:
for col in ['키', '몸무게', '구매옵션', '성별']:
    nunique = rv_df[col].nunique(dropna=False)
    unique_vals = rv_df[col].unique()
    print(f'[{col}]  고유값 수: {nunique}')
    print(f'  항목: {unique_vals}\n')

[키]  고유값 수: 70
  항목: [ 169  183  178  173  176  168  181  165  172  174  177  175  171  192
  180  164  154  160  170  162  184  187  156  185  166  155  158  163
  189  179   99  167  182  153  157  188  100  161  186  159  152  111
  190  150  200  123  148  110   69  122  151  199  149   40  191  220
  120    0 1172    1  193  116  130  147   61  195  999   70  135   85]

[몸무게]  고유값 수: 108
  항목: [  59   85   61   72   88   63   82   70   67   80   75   76   55  100
   60   48   68   50   64   78   65   57   62   73   58   49   84   74
   66   71   87   45   52   69   86   51   56   89   79   90   99   77
   53   93   96   83   30   98   43    5   95   92   81    2   38   44
   47   54   46   94   42  111  102  108   39  104   40  115  110  116
   97   91  150   10    4    6    3   33  123   11   22  122  105  117
  107    7  103   41  101   25  120    0   37    1  106  999  148  114
   17  113   35  109   20 4850  125    8   34   32]

[구매옵션]  고유값 수: 54
  항목: <StringArray>
[         

## 4. 요약

In [23]:
rv_num_cols = rv_df.select_dtypes(include='number').columns.tolist()
print('=' * 40)
print('  reviews_003.csv  기초 분석 요약')
print('=' * 40)
print(f'  행(row)          : {rv_df.shape[0]:,}')
print(f'  열(column)       : {rv_df.shape[1]}')
print(f'  수치형 컬럼      : {len(rv_num_cols)}개')
print(f'  카테고리 컬럼    : {len(rv_cat_cols)}개')
print(f'  결측값 있는 컬럼 : {rv_null_df[rv_null_df["결측 수"] > 0].shape[0]}개')
print(f'  전체 결측률      : {rv_df.isnull().mean().mean() * 100:.2f}%')
print(f'  중복 행          : {rv_dup_count:,}개')
print('=' * 40)

  reviews_003.csv  기초 분석 요약
  행(row)          : 30,767
  열(column)       : 14
  수치형 컬럼      : 6개
  카테고리 컬럼    : 6개
  결측값 있는 컬럼 : 4개
  전체 결측률      : 6.52%
  중복 행          : 0개


## 5. 추가 분석

In [24]:
print(rv_df['리뷰내용'].duplicated().sum())

# keep=False로 하면 첫 번째 포함 전부 True
print(rv_df['리뷰내용'].duplicated(keep=False).sum())

# 몇 개의 리뷰가 중복됐는지
print(rv_df[rv_df['리뷰내용'].duplicated(keep=False)]['리뷰내용'].nunique())

# 몇 개의 리뷰가 중복됐는지(NaN값 포함)
print(rv_df[rv_df['리뷰내용'].duplicated(keep=False)]['리뷰내용'].nunique(dropna=False))

print(rv_df[rv_df['리뷰내용'].duplicated(keep=False)]['리뷰내용'].isnull().sum())


2070
3673
1602
1603
50


첫 번째 2070 → "이 리뷰 나 전에 이미 나온 적 있어" 인 행의 수 → 쉽게 말해 복붙된 리뷰 수<br>
두 번째 3673 → 2070 + 원본(첫 등장)까지 포함한 수 → 중복에 엮인 행 전체 → 3673 - 2070 = 1603개가 원본(첫 등장)<br>
네 번째 1603 → 중복된 리뷰가 몇 가지 텍스트인지 (NaN 포함) → ex) "좋아요"라는 텍스트 1가지가 여러 번 쓰였으면 1로 카운트<br>
한 줄 요약하면, 1603가지 리뷰 문구가 반복 사용되어 총 3673개 행에 걸쳐 나타나고 있는 상황

In [25]:
for col in ['키', '몸무게']:
    print(f'[{col}]')
    print(rv_df[col].value_counts(dropna=False).sort_index(ascending=False))
    print()

[키]
키
1172      1
999       1
220      12
200      30
199       7
       ... 
69        5
61        1
40       15
1         1
0       900
Name: count, Length: 70, dtype: int64

[몸무게]
몸무게
4850      1
999       2
150      57
148       1
125       2
       ... 
4         3
3         7
2        52
1         3
0       953
Name: count, Length: 108, dtype: int64



In [26]:
sv_df = pd.read_csv('data/survey_003.csv')

In [27]:
sv_df.dtypes

goodsNo    int64
항목           str
답변           str
비율(%)      int64
응답수        int64
dtype: object